In [1]:
"""
plot_time_average.py
--------------------
Reads the time-averaged CSV produced by compute_time_average.py for each case
folder, plots the spatial intensity distribution as a heatmap, and saves the
figure to:
    <case_folder>/time_averaged_plot_intensity/<case_name>_time_average.png

Usage (Jupyter Notebook)
------------------------
Run each cell, or run the whole script with:
    %run plot_time_average.py
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_DIR        = r"D:\FCAI\plain_coupon\infared_data"

OUTPUT_SUBDIR   = "time_average"           # where compute_time_average.py saved results
OUTPUT_CSV      = "time_average.csv"

PLOT_SUBDIR     = "time_averaged_plot_intensity"   # where plots will be saved
PLOT_FORMAT     = "png"                            # "png", "pdf", "svg", etc.
DPI             = 150

# Colormap — "inferno" suits thermal/IR data; change freely
CMAP            = "inferno"

# Header lines to skip (same as in the averaging script)
HEADER_LINES    = 9
# ──────────────────────────────────────────────────────────────────────────────


def read_averaged_csv(filepath):
    """Read a time-average CSV and return the 2-D numpy data array."""
    with open(filepath, "r") as f:
        all_lines = f.readlines()

    # Skip header block and the blank separator line
    data_lines = [l.strip() for l in all_lines[HEADER_LINES + 1:] if l.strip()]
    data = np.array(
        [[float(v) for v in row.split(",")] for row in data_lines],
        dtype=np.float64,
    )
    return data


def parse_case_name(folder_name):
    """Convert e.g. 'cr_375_phi_09' → 'CR=375, φ=0.9' for the plot title."""
    # Expected pattern: cr_<cooling_rate>_phi_<equivalence*10>
    parts = folder_name.lower().split("_")
    try:
        cr_idx  = parts.index("cr")
        phi_idx = parts.index("phi")
        cr_val  = parts[cr_idx + 1]
        phi_raw = parts[phi_idx + 1]
        # Convert phi: "09" → 0.9, "10" → 1.0, "085" → 0.85, "118" → 1.18
        if len(phi_raw) == 2:
            phi_val = float(phi_raw) / 10
        else:
            phi_val = float(phi_raw) / 100
        return f"Cooling Rate = {cr_val}  |  φ = {phi_val:.2f}"
    except (ValueError, IndexError):
        return folder_name          # fall back to raw folder name


def plot_case(case_dir, case_name):
    """Load the time-average CSV for one case and save a heatmap figure."""
    csv_path = os.path.join(case_dir, OUTPUT_SUBDIR, OUTPUT_CSV)

    if not os.path.isfile(csv_path):
        print(f"  [SKIP] time_average CSV not found: {csv_path}")
        return

    data = read_averaged_csv(csv_path)
    n_rows, n_cols = data.shape

    # ── Create figure ────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(
        data,
        cmap=CMAP,
        aspect="auto",
        origin="upper",
        interpolation="nearest",
    )

    # Colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Intensity (Counts)", fontsize=11)
    cbar.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.0f"))

    # Labels & title
    ax.set_xlabel("Pixel Column", fontsize=11)
    ax.set_ylabel("Pixel Row", fontsize=11)
    title = parse_case_name(case_name)
    ax.set_title(f"Time-Averaged IR Intensity\n{title}", fontsize=12, fontweight="bold")

    # Annotate min/max
    vmin, vmax = data.min(), data.max()
    ax.text(
        0.01, 0.01,
        f"min = {vmin:.1f}   max = {vmax:.1f}   mean = {data.mean():.1f}",
        transform=ax.transAxes,
        fontsize=8,
        color="white",
        va="bottom",
    )

    plt.tight_layout()

    # ── Save ─────────────────────────────────────────────────────────────────
    plot_dir = os.path.join(case_dir, PLOT_SUBDIR)
    os.makedirs(plot_dir, exist_ok=True)
    out_path = os.path.join(plot_dir, f"{case_name}_time_average.{PLOT_FORMAT}")
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out_path}")


def main():
    base = os.path.normpath(BASE_DIR)
    print(f"Base directory:\n  {base}\n")

    if not os.path.isdir(base):
        print(f"ERROR: BASE_DIR does not exist or is not accessible:\n  {base}")
        print("  → Edit the BASE_DIR variable at the top of this script and re-run.")
        return

    entries   = sorted(os.listdir(base))
    case_dirs = [
        os.path.join(base, e) for e in entries
        if os.path.isdir(os.path.join(base, e))
    ]

    if not case_dirs:
        print(f"No sub-folders found in {base}")
        return

    print(f"Found {len(case_dirs)} case folder(s).\n")

    for case_dir in case_dirs:
        case_name = os.path.basename(case_dir)
        # Skip output folders that might sit alongside case folders
        if case_name in (OUTPUT_SUBDIR, PLOT_SUBDIR):
            continue
        print(f"[{case_name}]")
        plot_case(case_dir, case_name)

    print("\nAll plots done.")


main()

Base directory:
  D:\FCAI\plain_coupon\infared_data

Found 17 case folder(s).

[.ipynb_checkpoints]
  [SKIP] time_average CSV not found: D:\FCAI\plain_coupon\infared_data\.ipynb_checkpoints\time_average\time_average.csv
[anaconda_projects]
  [SKIP] time_average CSV not found: D:\FCAI\plain_coupon\infared_data\anaconda_projects\time_average\time_average.csv
[cr_1400_phi_085]
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_averaged_plot_intensity\cr_1400_phi_085_time_average.png
[cr_1400_phi_09]
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_09\time_averaged_plot_intensity\cr_1400_phi_09_time_average.png
[cr_1400_phi_10]
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_10\time_averaged_plot_intensity\cr_1400_phi_10_time_average.png
[cr_1400_phi_118]
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_118\time_averaged_plot_intensity\cr_1400_phi_118_time_average.png
[cr_1400_phi_137]
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_137\time_aver